<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/priprava_institucialnih_podatkov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import requests
import zipfile
import io

In [15]:
# Stolpci, ki jih potrebujemo
stolpci = ["UNITID", "INSTNM", "CONTROL", "ICLEVEL", "SECTOR"]

vsi_podatki = []
neuspesna_leta = []

# Prenos podatkov za vsa leta
for leto in range(2004, 2025):

    print("Prenašam leto:", leto)

    url = f"https://nces.ed.gov/ipeds/datacenter/data/HD{leto}.zip"

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:

            # Poiščemo CSV-datoteko
            ime_datoteke = [
                ime for ime in z.namelist()
                if ime.lower().endswith(".csv")
            ][0]

            # Preberemo podatke
            try:
                df = pd.read_csv(
                    z.open(ime_datoteke),
                    encoding="utf-8-sig",
                    low_memory=False
                )

            except UnicodeDecodeError:
                df = pd.read_csv(
                    z.open(ime_datoteke),
                    encoding="latin1",
                    low_memory=False
                )

        # Poenotimo imena stolpcev
        df.columns = (
            df.columns
            .str.replace("ï»¿", "", regex=False)
            .str.strip()
            .str.upper()
        )

        # Preverimo potrebne stolpce
        manjkajoci = [s for s in stolpci if s not in df.columns]

        if manjkajoci:
            print("Manjkajoči stolpci:", manjkajoci)
            neuspesna_leta.append(leto)
            continue

        # Izberemo stolpce in dodamo leto
        df = df[stolpci].copy()
        df["leto"] = leto

        vsi_podatki.append(df)

        print("Uspešno:", len(df), "institucij")

    except Exception as e:
        print("Napaka:", e)
        neuspesna_leta.append(leto)


# Preverimo, ali so bila uspešno obdelana vsa leta
if neuspesna_leta:
    raise ValueError(f"Neuspešna leta: {neuspesna_leta}")

# Združimo podatke
df_institucije = pd.concat(vsi_podatki, ignore_index=True)

# Uredimo po letu in instituciji
df_institucije = df_institucije.sort_values(
    ["leto", "UNITID"]
).reset_index(drop=True)

print("\nDimenzije zbirke:", df_institucije.shape)

print("\nŠtevilo različnih let:")
print(df_institucije["leto"].nunique())

print("\nŠtevilo zapisov po letih:")
print(df_institucije.groupby("leto").size())

Prenašam leto: 2004
Uspešno: 6916 institucij
Prenašam leto: 2005
Uspešno: 7018 institucij
Prenašam leto: 2006
Uspešno: 7052 institucij
Prenašam leto: 2007
Uspešno: 7052 institucij
Prenašam leto: 2008
Uspešno: 7126 institucij
Prenašam leto: 2009
Uspešno: 7316 institucij
Prenašam leto: 2010
Uspešno: 7503 institucij
Prenašam leto: 2011
Uspešno: 7643 institucij
Prenašam leto: 2012
Uspešno: 7735 institucij
Prenašam leto: 2013
Uspešno: 7764 institucij
Prenašam leto: 2014
Uspešno: 7687 institucij
Prenašam leto: 2015
Uspešno: 7647 institucij
Prenašam leto: 2016
Uspešno: 7521 institucij
Prenašam leto: 2017
Uspešno: 7153 institucij
Prenašam leto: 2018
Uspešno: 6857 institucij
Prenašam leto: 2019
Uspešno: 6559 institucij
Prenašam leto: 2020
Uspešno: 6440 institucij
Prenašam leto: 2021
Uspešno: 6289 institucij
Prenašam leto: 2022
Uspešno: 6256 institucij
Prenašam leto: 2023
Uspešno: 6163 institucij
Prenašam leto: 2024
Uspešno: 6072 institucij

Dimenzije zbirke: (147769, 6)

Število različnih let:


In [16]:
# Preverjanje kakovosti podatkov

print("Podvojene kombinacije UNITID in leto:")
print(df_institucije.duplicated(["UNITID", "leto"]).sum())

print("\nManjkajoče vrednosti:")
print(df_institucije[stolpci].isna().sum())

# Preverimo pričakovano obdobje
assert set(df_institucije["leto"]) == set(range(2004, 2025))

# Preverimo, da ni podvojenih institucij znotraj leta
assert not df_institucije.duplicated(["UNITID", "leto"]).any()

# Preverimo, da ključni stolpci niso prazni
assert df_institucije[stolpci].notna().all().all()

# Shranjevanje
df_institucije.to_csv(
    "IPEDS_institucije_2004_2024.csv",
    index=False
)

print("\nDatoteka uspešno shranjena!")

Podvojene kombinacije UNITID in leto:
0

Manjkajoče vrednosti:
UNITID     0
INSTNM     0
CONTROL    0
ICLEVEL    0
SECTOR     0
dtype: int64

Datoteka uspešno shranjena!
